# Data Science Bootcamp Avito

## 1. Библиотеки и настройки

In [ ]:
from pathlib import Path
import json
import platform
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

from metric import precision_at_recall

seeds = [42, 43, 44]
validation_start = pd.Timestamp("2026-04-17", tz="UTC")

## 2. Данные

In [65]:
text_columns = [
    "platform", "user_agent", "item_id", "item_category",
    "item_location", "seller_type", "search_query",
]
meta_columns = ["cookie_id", "cookie_created_at", "window_start_ts", "window_end_ts"]
cat_features = [
    "platform_mode", "user_agent_mode", "item_category_mode",
    "item_location_mode", "seller_type_mode",
]

train = pd.read_csv("data/train.csv", dtype={"cookie_id": str})
test = pd.read_csv("data/test.csv", dtype={"cookie_id": str})
events = pd.read_csv(
    "data/events.csv.gz",
    dtype=dict.fromkeys(["cookie_id", "eid", "event_name"] + text_columns, str),
    low_memory=False,
)
for frame in [train, test]:
    for column in meta_columns[1:]:
        frame[column] = pd.to_datetime(frame[column], format="mixed", utc=True)
events["event_ts"] = pd.to_datetime(events["event_ts"], format="mixed", utc=True)

print(f"Train: {len(train):,}; test: {len(test):,}; событий: {len(events):,}")
print(f"Доля ботов в train: {train.target.mean():.2%}")
display(train.head())

Train: 11,091; test: 4,909; событий: 328,905
Доля ботов в train: 8.11%


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41+00:00,2026-04-06 00:00:00+00:00,2026-04-07 00:00:00+00:00,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24+00:00,2026-04-06 00:00:00+00:00,2026-04-07 00:00:00+00:00,0
2,ck_9320229ef6304522,2026-03-04 00:08:02+00:00,2026-04-06 00:00:00+00:00,2026-04-07 00:00:00+00:00,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40+00:00,2026-04-06 00:00:00+00:00,2026-04-07 00:00:00+00:00,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35+00:00,2026-04-06 00:00:00+00:00,2026-04-07 00:00:00+00:00,0


## 3. Окно наблюдения

Оставляем только события внутри окна куки. Иначе в признаки может попасть информация из будущего. Полные повторы строк удаляем. Считаем их повторной записью одного события, хотя без `event_id` это остаётся допущением.

In [66]:
meta = pd.concat([train[meta_columns], test[meta_columns]]).set_index("cookie_id")
assert (meta.window_end_ts > meta.window_start_ts).all()
assert (meta.cookie_created_at <= meta.window_end_ts).all()

window_events = events.merge(
    meta[["window_start_ts", "window_end_ts"]],
    left_on="cookie_id", right_index=True, validate="many_to_one",
)
inside_window = (
    (window_events.event_ts >= window_events.window_start_ts)
    & (window_events.event_ts < window_events.window_end_ts)
)
window_events = window_events.loc[inside_window].copy()
ev = window_events.drop_duplicates().copy()

print(f"Событий внутри окна: {len(window_events):,}")
print(f"Удалено полных дублей: {len(window_events) - len(ev):,}")
print(f"Осталось событий для признаков: {len(ev):,}")

Событий внутри окна: 288,126
Удалено полных дублей: 4,293
Осталось событий для признаков: 283,833


## 4. Подготовка полей

Приведём варианты написания платформ и запросов к одному виду. Пустые строки считаем пропусками, координаты и номера страниц - числами

In [67]:
for column in text_columns:
    ev[column] = ev[column].astype("string").replace("", pd.NA)

ev["platform"] = ev.platform.str.strip().str.lower().replace(
    {"desktop": "web", "iphone": "ios"}
)
ev["search_query"] = (
    ev.search_query.str.strip().str.lower()
    .str.replace(r"\s+", " ", regex=True).replace("", pd.NA)
)
for column in ["pointer_x", "pointer_y", "search_page"]:
    ev[column] = pd.to_numeric(ev[column], errors="coerce").replace([np.inf, -np.inf], np.nan)

Полный user_agent слишком подробный. Оставим семейство клиента и операционную систему. Признаки headless и программных клиентов добавим отдельно - это признаки для модели, а не готовое правило "такой клиент всегда бот"

In [79]:
ua = ev.user_agent.fillna("__MISSING__").str.strip().str.lower()
browser_patterns = {
    "headlesschrome": "headless_chrome", "python-requests": "python_requests",
    "python-urllib": "python_urllib", "scrapy": "scrapy", "curl/": "curl",
    "go-http-client": "go_http_client", "node-fetch": "node_fetch",
    "avito/": "avito_app", "yabrowser": "yandex", "edg/": "edge",
    "opr/": "opera", "firefox/|fxios/": "firefox", "chrome/|crios/": "chrome",
    "safari/": "safari", "okhttp/": "okhttp",
}
os_patterns = {
    "android": "android", "iphone|ipad|ios ": "ios", "windows": "windows",
    "macintosh|mac os": "macos", "linux": "linux",
}
browser = np.select(
    [ua.str.contains(pattern).to_numpy(dtype=bool) for pattern in browser_patterns],
    list(browser_patterns.values()), default="other",
)
os_name = np.select(
    [ua.str.contains(pattern).to_numpy(dtype=bool) for pattern in os_patterns],
    list(os_patterns.values()), default="unknown",
)
ev["user_agent"] = pd.Series(browser, index=ev.index) + "|" + os_name
ev.loc[ua.isin(["", "__missing__"]), "user_agent"] = pd.NA

automation_pattern = (
    "headless|python-requests|python-urllib|scrapy|curl/|go-http-client|node-fetch|selenium|phantomjs"
)
ev["ua_automation"] = ua.str.contains(automation_pattern).astype(float)
ev["ua_headless"] = ua.str.contains("headless", regex=False).astype(float)

In [80]:
ev.ua_automation[ev.ua_automation > 0]

520       1.0
521       1.0
522       1.0
523       1.0
524       1.0
         ... 
281638    1.0
281639    1.0
282224    1.0
282225    1.0
282226    1.0
Name: ua_automation, Length: 11259, dtype: float64

## 5. Частота и разнообразие действий

Дальше одна строка таблицы признаков будет соответствовать одной куке. Начнём с возраста куки, количества событий и числа разных значений в полях. Например, доля уникальных объявлений помагает отличить повторные просмотры от обхода каталога

In [81]:
ev["event_type"] = (
    ev.event_name.astype("string").replace("", pd.NA)
    .fillna("eid=" + ev.eid.astype("string")).fillna("__MISSING__").astype(str)
)
ev = ev.sort_values(["cookie_id", "event_ts", "event_type"], kind="stable").reset_index(drop=True)
groups = ev.groupby("cookie_id", sort=False)
n_events = groups.size().reindex(meta.index, fill_value=0)
event_denominator = n_events.replace(0, np.nan)
window_seconds = (meta.window_end_ts - meta.window_start_ts).dt.total_seconds()

features = pd.DataFrame(index=meta.index)
features["window_hours"] = window_seconds / 3600
features["cookie_age_at_end_hours"] = (meta.window_end_ts - meta.cookie_created_at).dt.total_seconds() / 3600
features["cookie_age_at_start_hours"] = (
    (meta.window_start_ts - meta.cookie_created_at).dt.total_seconds().clip(lower=0) / 3600
)
features["created_in_window"] = (meta.cookie_created_at >= meta.window_start_ts).astype(int)
features["n_events"] = n_events
features["has_events"] = (n_events > 0).astype(int)
features["events_per_window_hour"] = n_events / features.window_hours

for column in text_columns + ["event_type"]:
    nonmissing = groups[column].count().reindex(meta.index, fill_value=0)
    unique = groups[column].nunique().reindex(meta.index, fill_value=0)
    features[f"{column}_nunique"] = unique
    features[f"{column}_unique_ratio"] = unique / nonmissing.replace(0, np.nan)
    features[f"{column}_missing_fraction"] = 1 - nonmissing / event_denominator

Одного числа уникальных значений мало. Посмотрим также, насколько часто повторяется самое популярное значение и насколько разнообразны остальные. Для разнообразия используем энтропию: она равна нулю, если значение всегда одно, и растёт при более равномерном распределении

In [82]:
for column in [
    "event_type", "platform", "user_agent", "item_category",
    "item_location", "seller_type", "search_query", "item_id",
]:
    counts = ev.groupby(["cookie_id", column], sort=True).size()
    shares = counts / counts.groupby(level=0).transform("sum")
    features[f"{column}_entropy"] = (-shares * np.log2(shares)).groupby(level=0).sum()
    features[f"{column}_top_share"] = shares.groupby(level=0).max()
    if f"{column}_mode" in cat_features:

        mode = counts.groupby(level=0).idxmax().str[1]
        features[f"{column}_mode"] = mode.reindex(meta.index).fillna("__MISSING__")

features["ua_automation_fraction"] = groups.ua_automation.mean()
features["ua_headless_fraction"] = groups.ua_headless.mean()

## 6. Время и паузы

У двух кук может быть одинаковое число просмотров, но совершенно разный ритм. Посчитаем длительность активности и паузы между событиями (среднее, разброс, квантили и долю коротких интервалов)

In [83]:
first_event = groups.event_ts.min()
last_event = groups.event_ts.max()
features["active_span_seconds"] = (last_event - first_event).dt.total_seconds()
features["first_event_delay_seconds"] = (first_event - meta.window_start_ts).dt.total_seconds()
features["last_event_silence_seconds"] = (meta.window_end_ts - last_event).dt.total_seconds()
features["events_per_active_minute"] = n_events * 60 / (features.active_span_seconds + 1)

ev["gap_seconds"] = groups.event_ts.diff().dt.total_seconds()
gaps = ev.groupby("cookie_id").gap_seconds
features = features.join(gaps.agg(["mean", "std", "min", "max", "median"]).add_prefix("gap_seconds_"))
for q in [0.1, 0.25, 0.75, 0.9]:
    features[f"gap_seconds_q{int(q * 100):02}"] = gaps.quantile(q)

n_gaps = gaps.count().replace(0, np.nan)
for seconds in [0, 1, 5, 10, 60]:
    short_gaps = ev.gap_seconds.le(seconds).groupby(ev.cookie_id).sum()
    features[f"gap_le_{seconds}s_fraction"] = short_gaps / n_gaps
features["gap_coefficient_of_variation"] = features.gap_seconds_std / (features.gap_seconds_mean + 1e-6)

Регулярные паузы могут быть полезным признаком автоматизации

In [84]:
positive_gaps = ev.loc[ev.gap_seconds > 0]
positive_stats = positive_gaps.groupby("cookie_id").gap_seconds.agg(["mean", "std", "median"])
features = features.join(positive_stats.add_prefix("positive_gap_"))

counts = positive_gaps.groupby(["cookie_id", "gap_seconds"]).size()
shares = counts / counts.groupby(level=0).transform("sum")
features["positive_gap_top_share"] = shares.groupby(level=0).max()
features["positive_gap_entropy"] = (-shares * np.log2(shares)).groupby(level=0).sum()
features["positive_gap_cv"] = features.positive_gap_std / (features.positive_gap_mean + 1e-6)
features["gap_iqr_over_median"] = (
    (features.gap_seconds_q75 - features.gap_seconds_q25) / (features.gap_seconds_median + 1)
)

for event_type in ["search_results_view", "item_view"]:
    subset = ev.loc[ev.event_type.eq(event_type)].copy()
    subset["gap"] = subset.groupby("cookie_id").event_ts.diff().dt.total_seconds()
    stats = subset.groupby("cookie_id").gap.agg(["mean", "std", "median"])
    features = features.join(stats.add_prefix(f"{event_type}_gap_"))
    features[f"{event_type}_gap_cv"] = stats["std"] / (stats["mean"] + 1e-6)

## 7. Активность в течение суток

Разобьём окно на 24 части и посчитаем долю событий в каждой. Это покажет, распределены ли действия по всему дню или собраны в коротком промежутке. Затем посмотрим нагрузку в активные минуты и длину сессий

In [85]:
window_length = (ev.window_end_ts - ev.window_start_ts).dt.total_seconds()
position_in_window = (ev.event_ts - ev.window_start_ts).dt.total_seconds() / window_length
ev["window_bin"] = (position_in_window * 24).astype(int).clip(0, 23)
hourly = ev.groupby(["cookie_id", "window_bin"]).size().unstack(fill_value=0)
hourly = hourly.reindex(index=meta.index, columns=range(24), fill_value=0)
hourly_shares = hourly.div(event_denominator, axis=0).fillna(0)
features = features.join(hourly_shares.rename(columns=lambda hour: f"window_bin_{hour:02}_share"))
features["active_window_bins"] = hourly.gt(0).sum(axis=1)
features["window_bin_top_share"] = hourly_shares.max(axis=1)
features["window_bin_entropy"] = -(hourly_shares * np.log2(hourly_shares.replace(0, 1))).sum(axis=1)

ev["minute"] = ev.event_ts.dt.floor("min")
per_minute = ev.groupby(["cookie_id", "minute"]).size()
features["active_minutes"] = per_minute.groupby(level=0).size().reindex(meta.index, fill_value=0)
features["events_per_nonempty_minute_max"] = per_minute.groupby(level=0).max().reindex(meta.index, fill_value=0)
features["events_per_nonempty_minute_mean"] = per_minute.groupby(level=0).mean()

In [86]:
# Новая сессия начинается после паузы больше 30 минут
new_session = ev.gap_seconds.isna() | ev.gap_seconds.gt(1800)
ev["session"] = new_session.groupby(ev.cookie_id).cumsum()
sessions = ev.groupby(["cookie_id", "session"]).agg(
    n_events=("event_ts", "size"), first=("event_ts", "min"), last=("event_ts", "max"),
)
features["n_sessions_30m"] = sessions.groupby(level=0).size().reindex(meta.index, fill_value=0)
features["session_events_mean"] = sessions.n_events.groupby(level=0).mean()
features["session_events_max"] = sessions.n_events.groupby(level=0).max().reindex(meta.index, fill_value=0)
session_seconds = (sessions["last"] - sessions["first"]).dt.total_seconds()
features["session_seconds_mean"] = session_seconds.groupby(level=0).mean()
features["session_seconds_max"] = session_seconds.groupby(level=0).max()

## 8. Поиск и движение курсора

В поиске учитываем глубину просмотра выдачи и длину запросов. Для курсора считаем число разных координат, разброс, расстояния и скорость перемещения

In [87]:
page_stats = ev.groupby("cookie_id").search_page.agg(["mean", "std", "min", "max", "median"])
features = features.join(page_stats.add_prefix("search_page_"))
n_pages = ev.search_page.notna().groupby(ev.cookie_id).sum()
features["search_page_present_fraction"] = n_pages / event_denominator
for page in [2, 5, 10]:
    deep_pages = ev.search_page.ge(page).groupby(ev.cookie_id).sum()
    features[f"search_page_ge_{page}_fraction"] = deep_pages / n_pages.replace(0, np.nan)

ev["query_length"] = ev.search_query.str.len().astype(float)
ev["query_words"] = ev.search_query.str.split().str.len().astype("Float64").astype(float)
query_stats = ev.groupby("cookie_id").agg(
    query_length_mean=("query_length", "mean"), query_length_max=("query_length", "max"),
    query_length_std=("query_length", "std"), query_words_mean=("query_words", "mean"),
    query_words_max=("query_words", "max"),
)
features = features.join(query_stats).copy()

In [88]:
pointer = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()
pointer_count = pointer.groupby("cookie_id").size().reindex(meta.index, fill_value=0)
unique_pointer = pointer.drop_duplicates(["cookie_id", "pointer_x", "pointer_y"])
features["pointer_pairs_count"] = pointer_count
features["pointer_pairs_nunique"] = unique_pointer.groupby("cookie_id").size().reindex(meta.index, fill_value=0)
features["pointer_present_fraction"] = pointer_count / event_denominator
features["pointer_unique_ratio"] = features.pointer_pairs_nunique / pointer_count.replace(0, np.nan)

for coordinate in ["pointer_x", "pointer_y"]:
    stats = pointer.groupby("cookie_id")[coordinate].agg(["mean", "std", "min", "max"])
    features = features.join(stats.add_prefix(f"{coordinate}_"))
    features[f"{coordinate}_range"] = stats["max"] - stats["min"]

pointer = pointer.sort_values(["cookie_id", "event_ts", "pointer_x", "pointer_y"], kind="stable")
steps = pointer.groupby("cookie_id")
dx = steps.pointer_x.diff()
dy = steps.pointer_y.diff()
dt = steps.event_ts.diff().dt.total_seconds()

# Для событий с одинаковым временем порядок неизвестен: скорость по ним не считаем.
pointer["distance"] = np.hypot(dx, dy).where(dt > 0)
pointer["speed"] = pointer.distance / dt.where(dt > 0)
for column in ["distance", "speed"]:
    stats = pointer.groupby("cookie_id")[column].agg(["mean", "std", "max"])
    features = features.join(stats.add_prefix(f"pointer_{column}_"))
valid_steps = pointer.distance.notna().groupby(pointer.cookie_id).sum()
stationary_steps = pointer.distance.eq(0).groupby(pointer.cookie_id).sum()
features["pointer_stationary_fraction"] = stationary_steps / valid_steps.replace(0, np.nan)

## 9. Матрица для модели

Осталось добавить количество и долю событий каждого типа. Список типов берём только из обучающей части; новые события попадут в группу `__OTHER__`

Эта подготовка нужна дважды: для временной валидации и для финального обучения. Поэтому вынесем её в небольшую функцию. Числовые пропуски оставим как `NaN` - CatBoost умеет с ними работать. Категориальные пропуски заменим отдельной строкой

In [89]:
event_counts = ev.groupby(["cookie_id", "event_type"]).size().unstack(fill_value=0)
event_counts = event_counts.reindex(meta.index, fill_value=0).sort_index(axis=1)


def make_matrix(fitting_ids):
    counts_in_fit = event_counts.loc[fitting_ids].sum()
    event_types = counts_in_fit.index[counts_in_fit > 0]
    counts = event_counts.loc[:, event_types]
    event_features = pd.DataFrame(index=meta.index)
    for event_type in event_types:
        event_features[f"event_count::{event_type}"] = counts[event_type]
        event_features[f"event_share::{event_type}"] = (counts[event_type] / event_denominator).fillna(0)
    other = event_counts.sum(axis=1) - counts.sum(axis=1)
    event_features["event_count::__OTHER__"] = other
    event_features["event_share::__OTHER__"] = (other / event_denominator).fillna(0)
    matrix = features.join(event_features)
    matrix[cat_features] = matrix[cat_features].fillna("__MISSING__").astype(str)
    numeric = matrix.columns.difference(cat_features)
    matrix[numeric] = matrix[numeric].astype(float).replace([np.inf, -np.inf], np.nan)
    columns = matrix.loc[fitting_ids].nunique(dropna=False).gt(1)
    return matrix.loc[:, columns]

## 10. Валидация

Тест начинается позже train. Поэтому проверяем модель на последних трёх днях train:
17–19 апреля

In [90]:
fit_mask = (train.window_start_ts < validation_start) & (train.window_end_ts <= validation_start)
valid_mask = train.window_start_ts >= validation_start
fit_ids = train.loc[fit_mask, "cookie_id"].tolist()
valid_ids = train.loc[valid_mask, "cookie_id"].tolist()
y = train.set_index("cookie_id").target
assert y.loc[fit_ids].nunique() == y.loc[valid_ids].nunique() == 2
assert train.window_end_ts.max() <= test.window_start_ts.min()

matrix = make_matrix(fit_ids)
X_fit = matrix.loc[fit_ids]
X_valid = matrix.loc[valid_ids]
print(f"Обучение: {len(X_fit):,} кук; валидация: {len(X_valid):,}; признаков: {X_fit.shape[1]}")

Обучение: 9,140 кук; валидация: 1,951; признаков: 167


In [92]:
baseline_features = window_events.groupby("cookie_id").agg(
    n_events=("event_ts", "size"), item_nunique=("item_id", "nunique"),
).reindex(train.cookie_id).fillna(0)
baseline = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0, n_jobs=4)
baseline.fit(baseline_features.loc[fit_ids], y.loc[fit_ids])
baseline_score = baseline.predict_proba(baseline_features.loc[valid_ids])[:, 1]
baseline_metric = precision_at_recall(y.loc[valid_ids], baseline_score)
print(f"Baseline: P@R≥0.7 = {baseline_metric:.4f}")

Baseline: P@R≥0.7 = 0.1025


## 11. CatBoost

Используем три модели с одинаковыми настройками и разными seeds, затем усредним их предсказания

In [93]:
model_params = dict(
    iterations=700,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=7,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    thread_count=4,
    task_type="CPU",
    allow_writing_files=False,
    verbose=False,
)


def fit_ensemble(X_train, y_train, X_predict):
    models = []
    scores = np.zeros(len(X_predict))
    categories = [column for column in cat_features if column in X_train.columns]
    for seed in seeds:
        model = CatBoostClassifier(**model_params, random_seed=seed)
        model.fit(X_train, y_train, cat_features=categories)
        scores += model.predict_proba(X_predict)[:, 1] / len(seeds)
        models.append(model)
        print(f"Готова модель с seed={seed}")
    return scores, models

In [95]:
valid_score, validation_models = fit_ensemble(X_fit, y.loc[fit_ids], X_valid)
validation_metric = precision_at_recall(y.loc[valid_ids], valid_score)

print(f"Baseline: P@R≥0.7 = {baseline_metric:.4f}")
print(f"CatBoost: P@R≥0.7 = {validation_metric:.4f}")

Готова модель с seed=42
Готова модель с seed=43
Готова модель с seed=44
Baseline: P@R≥0.7 = 0.1025
CatBoost: P@R≥0.7 = 0.8058


## 12. Финальное обучение

Теперь используем весь train. Словарь событий, постоянные признаки и веса классов пересчитываются по всем обучающим кукам. Настройки CatBoost остаются прежними

In [96]:
train_ids = train.cookie_id.tolist()
test_ids = test.cookie_id.tolist()
final_matrix = make_matrix(train_ids)
X_train = final_matrix.loc[train_ids]
X_test = final_matrix.loc[test_ids]
test_score, final_models = fit_ensemble(X_train, y.loc[train_ids], X_test)

Готова модель с seed=42
Готова модель с seed=43
Готова модель с seed=44


## 13. Сохранение результата

In [97]:
submission = pd.DataFrame({"cookie_id": test_ids, "score": test_score})
submission.to_csv("submission.csv", index=False, float_format="%.17g")

display(submission.head())

,cookie_id,score
0,ck_315fb710a0e371e7,0.000685
1,ck_a76ee3b3e3e522fd,0.167420
2,ck_94c9a4d382689e82,0.151438
3,ck_8eaf9509ad9462a0,0.001037
4,ck_9a88a5a989cb5bc6,0.018701
